# S8 · Modelos cerrados
Mismo problema y particiones. Google Cloud está pospuesto: este notebook prepara datos y especificación sin crear recursos. Las llamadas de inferencia se activan explícitamente.

**Recorrido:** reutilizar datos → convertir al formato Google → describir trabajo → ejecutar solo con laboratorio autorizado → evaluar y comparar. Las celdas de esta versión inspeccionan datos y especificación. Los comandos remotos se muestran como comentarios, por lo que ejecutar la celda no los envía.

In [ ]:
# Path permite construir rutas, comprobar archivos y leer/escribir sin concatenar separadores a mano.
from pathlib import Path

# Gestiona rutas de importación y el intérprete actual de Python.
# Convierte entre texto JSON y diccionarios/listas Python, manteniendo respuestas como evidencia.
import sys, json

# Raíz del proyecto utilizada para resolver las rutas de datos y scripts.
ROOT = Path.cwd()
# Si Jupyter se abrió dentro de notebooks/, la raíz es la carpeta superior.
if ROOT.name == "notebooks":
    # .parent es la carpeta que contiene a notebooks/.
    ROOT = ROOT.parent
# Añade una ruta de importación para poder reutilizar el paquete curso y los scripts del proyecto.
sys.path.insert(0, str(ROOT))
# Funciones compartidas del curso: experimento congelado y lectura de JSONL.
from curso.common import assert_frozen, read_jsonl

# Comprueba que datos, prompt, revisión y parámetros coinciden con el experimento congelado.
assert_frozen()

## 1. Una petición, cualquier proveedor
Los comandos se muestran sin ejecutarlos automáticamente. Se conserva el mismo caso de desarrollo.

`demo.py` es el mismo punto de entrada de S7. Sin `--proveedor` usa `LLM_PROVIDER` y `LLM_MODEL` de `.env`: la ruta del curso es Claude Haiku 5.5 por la API de Anthropic, y basta con cambiar esas variables para usar OpenAI, OpenRouter, Gemini o cualquier API compatible. Cada ruta devuelve su propia estructura: Anthropic `content`/`stop_reason`/`usage`, Chat Completions `choices`/`usage` y Gemini `candidates`/`usageMetadata`. Conservar ruta y modelo es necesario para interpretar parámetros y consumo. Una llamada de inferencia a Gemini no crea un modelo ajustado.

In [ ]:
# Ruta de .env (por defecto, Claude Haiku 5.5 por la API de Anthropic):
# python scripts/demo.py --caso DE-01
# python scripts/demo.py --caso DE-01 --fewshot
# Otra ruta sin cambiar .env:
# python scripts/demo.py --proveedor gemini --modelo gemini-3.1-flash-lite --caso DE-01
# python scripts/demo.py --proveedor openrouter --modelo google/gemini-3.1-flash-lite --caso DE-01

In [ ]:
# Lee .env y muestra qué proveedor y modelo usarán demo.py y evaluar_remotos.py sin --proveedor.
# route no crea clientes ni llama a ninguna API; las rutas disponibles están en PROVEEDORES.
from curso.remote import route, PROVEEDORES

# Nombres de ruta válidos para LLM_PROVIDER o --proveedor.
print(sorted(PROVEEDORES))
# Tupla (proveedor, modelo) que se usaría sin argumentos; falla si .env pide una ruta sin modelo.
route()

## 2. Conversión JSONL
System y contents user/model. Los IDs se mantienen aparte. Test no se sube para el trabajo.

`gemini_record` conserva las instrucciones comunes y convierte incidencia a role=user y objetivo a role=model, ambos dentro de contents/parts. El ID no se envía como conversación: `ids.json` permite relacionar cada línea con su registro original. Esta conversión es local y no sube el test.

In [ ]:
# Conversión de una fila S7 al formato systemInstruction + contents de Gemini.
from curso.remote import gemini_record

# Primer ejemplo de train: id y familia no pasan al registro convertido.
row = read_jsonl(ROOT / "datos/train.jsonl")[0]
# Muestra el registro convertido bajo la celda.
gemini_record(row)

In [ ]:
# ids.json relaciona cada línea convertida con su ID original.
ids = json.loads((ROOT / "datos/gemini/ids.json").read_text(encoding="utf-8"))
# Cuántas líneas tiene cada partición convertida (240/30/30).
{k: len(v["ids_por_linea"]) for k, v in ids.items()}

## 3. Especificación offline
Nombres de ejemplo NO creados. Preparar no entrena; hace falta un proyecto/permiso/bucket/presupuesto reales.

`job_spec` valida nombres y devuelve un diccionario con modelo, región, train, validación y configuración. Los valores ejemplo-no-creado son marcadores didácticos. No hay cliente Google en esa función y llamar a job_spec no inicia entrenamiento ni crea un bucket.

In [ ]:
# Añade scripts/ a las rutas de importación para reutilizar job_spec de gemini_tuning.py.
sys.path.insert(0, str(ROOT / "scripts"))
# Función que describe el trabajo de tuning sin llamar a Google.
from gemini_tuning import job_spec

# Solo describe el trabajo con nombres de ejemplo: no llama a Google ni crea recursos.
job_spec("ejemplo-no-creado", "ejemplo-no-creado")

## 4. Crear, consultar, predecir
Seguir docs/modelos-remotos.md cuando se prepare el laboratorio. Consultar el mismo job no crea otro. No afirmar que un prompt diferente es fine-tuning.

`subir-datos` escribe solo train/validación en un bucket existente. `crear` solicita tuning y guarda su ID. `estado` consulta ese ID para seguir el mismo trabajo. Proyecto, permisos y estimación/presupuesto deben estar configurados; el presupuesto anotado no impone un tope técnico al proveedor.

In [ ]:
# Solo con laboratorio y presupuesto revisados:
# python scripts/gemini_tuning.py subir-datos ...
# python scripts/gemini_tuning.py crear ...
# python scripts/gemini_tuning.py estado --proyecto ... --job ...
# python scripts/demo.py --proveedor google --modelo RECURSO_MODELO_AJUSTADO --caso DE-01

## 5. Comparación y límites
Las filas Google pendientes se conservan. Gemini por OpenRouter es adicional. Tokens/coste ausentes no se convierten en cero.

Para comparar hay que conservar datos, prompt y validador. La pareja Gemini base/ajustado se mide por su ruta Google; Claude usa prompting/few-shot por la ruta de `.env` (la del curso: Anthropic). El informe de `resumir.py` indica qué modelo y qué ruta se usaron en cada condición. Los registros que no se han ejecutado siguen pendientes. `None` en tokens o coste indica que no hay medida disponible.

In [ ]:
# Evidencia histórica identificada por fecha y limitaciones; no es una ejecución nueva del notebook.
reference = json.loads(
    (ROOT / "docs/resultados-referencia.json").read_text(encoding="utf-8")
)
# Fecha y limitaciones de aquella ejecución.
print("Referencia anterior:", reference["fecha"], reference["limitaciones"])
# Presenta resultados históricos identificados; no los atribuye a una ejecución de esta celda.
reference["resultados"]